# Load packages and libraries

In [1]:
.libPaths()
assign(".lib.loc", "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library", envir = environment(.libPaths))
.libPaths()
# sessionInfo()


suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(dplyr)) 
suppressMessages(library(ggplot2)) 
suppressMessages(library(Matrix)) 
suppressMessages(library(data.table)) 
suppressMessages(library(ggpubr)) 
suppressMessages(library(ggplot2))
suppressMessages(library(pheatmap))
suppressMessages(library(presto))
suppressMessages(library("qlcMatrix"))
suppressMessages(library("cowplot"))
suppressMessages(library("RColorBrewer"))
suppressMessages(library("plyr"))
suppressMessages(library("forcats"))
suppressMessages(library('ggeasy'))
suppressMessages(library('dplyr'))
suppressMessages(library("svglite"))
suppressMessages(library("ape"))
suppressMessages(library("ggforce"))
suppressMessages(library("tidyr"))
suppressMessages(library("edgeR"))
suppressMessages(library("apeglm"))
suppressMessages(library("DESeq2"))
suppressMessages(library("tibble")) 
suppressMessages(library(future))
suppressMessages(library(future))


library("optparse")
suppressMessages(library("splitstackshape")) 
suppressMessages(library("ggupset"))

library("org.Mm.eg.db", lib.loc="/home/manuel.tardaguila/R/x86_64-pc-linux-gnu-library/4.1/")

suppressMessages(library("ggrepel"))



[1] "/group/soranzo/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

Loading required package: AnnotationDbi

Warning message:
“replacing previous import ‘utils::findMatches’ by ‘S4Vectors::findMatches’ when loading ‘AnnotationDbi’”

Attaching package: ‘AnnotationDbi’


The following object is masked from ‘package:dplyr’:

    select






# Read in featureCounts files and transform to matrix structure

In [2]:
setwd("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/")

list_files<-dir()

list_files_sel<-list_files[grep("_Quant.genes.results$", list_files)]

str(list_files_sel)
cat("\n")

sample_vector<-gsub("_Quant.genes.results$","",list_files_sel)

str(sample_vector)
cat("\n")

df<-as.data.frame(cbind(list_files_sel,sample_vector))
colnames(df)<-c('file','sample')

df$file<-paste("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/",df$file, sep='')


str(df)
cat("\n")




 chr [1:30] "LM411_R1_t0_Quant.genes.results" ...

 chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...

'data.frame':	30 obs. of  2 variables:
 $ file  : chr  "/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/LM411_R1_t0_Quant.genes.results" "/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/LM411_R1_t1_Quant.genes.results" "/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/LM411_R1_t2_Quant.genes.results" "/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/LM411_R1_t3_Quant.genes.results" ...
 $ sample: chr  "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



In [3]:
DEBUG<-0

# Initialize a variable to store the merged count matrix (optional, but good practice)
all_raw_counts <- NULL 


for(i in 1:dim(df)[1]){

    sample_array_sel<-df$sample[i]

    cat("-------------------------->\t")
    cat(sprintf(as.character(sample_array_sel)))
    cat("\n")

    raw_counts_file<-df$file[i]

    raw_counts_df<-read.table(file=raw_counts_file,sep="\t", header=TRUE, stringsAsFactors = FALSE)
    #colnames(raw_counts_df)[8]<-sample_array_sel


    if(DEBUG == 1){
        
        cat("raw_counts_df_0\n")
        str(raw_counts_df)
        cat("\n")
    }


    indx_sel<-which(colnames(raw_counts_df)%in%c('gene_id','expected_count'))


    raw_counts_df_subset<-raw_counts_df[,indx_sel]

    

    if(DEBUG == 1){
        
        cat("raw_counts_df_subset_0\n")
        str(raw_counts_df_subset)
        cat("\n")
    }

    colnames(raw_counts_df_subset)[which(colnames(raw_counts_df_subset) == 'expected_count')]<-sample_array_sel

    # 3. Initialize or Merge the count matrix
    if (is.null(all_raw_counts)) {
        # If this is the first sample, initialize the matrix
        all_raw_counts <- raw_counts_df_subset
        } else {
            # For subsequent samples, merge the new count column 
            # using 'gene_id' as the key to align the rows.
            all_raw_counts <- merge(
                x = all_raw_counts, 
                y = raw_counts_df_subset, 
                by.x = "gene_id", 
                by.y = "gene_id",
                all = TRUE # Keeps all Gene IDs, even if a sample has a count of 0 for a gene.
            )
        }#if (is.null(all_raw_counts)

    if(DEBUG == 1){
        
        cat("all_raw_counts_0\n")
        str(all_raw_counts)
        cat("\n")
    }

    
}#i in 1:length(sample_array)


-------------------------->	LM411_R1_t0
-------------------------->	LM411_R1_t1
-------------------------->	LM411_R1_t2
-------------------------->	LM411_R1_t3
-------------------------->	LM411_R1_t4
-------------------------->	LM411_R2_t0
-------------------------->	LM411_R2_t1
-------------------------->	LM411_R2_t2
-------------------------->	LM411_R2_t3
-------------------------->	LM411_R2_t4
-------------------------->	LM411_R3_t0
-------------------------->	LM411_R3_t1
-------------------------->	LM411_R3_t2
-------------------------->	LM411_R3_t3
-------------------------->	LM411_R3_t4
-------------------------->	LM511_R1_t0
-------------------------->	LM511_R1_t1
-------------------------->	LM511_R1_t2
-------------------------->	LM511_R1_t3
-------------------------->	LM511_R1_t4
-------------------------->	LM511_R2_t0
-------------------------->	LM511_R2_t1
-------------------------->	LM511_R2_t2
-------------------------->	LM511_R2_t3
-------------------------->	LM511_R2_t4


In [4]:
cat("all_raw_counts_0\n")
str(all_raw_counts)
cat("\n")

all_raw_counts_0
'data.frame':	78334 obs. of  31 variables:
 $ gene_id    : chr  "ENSMUSG00000000001" "ENSMUSG00000000003" "ENSMUSG00000000028" "ENSMUSG00000000031" ...
 $ LM411_R1_t0: num  5384 0 58 0 16 ...
 $ LM411_R1_t1: num  8039 0 912 0 53 ...
 $ LM411_R1_t2: num  6724 0 1953 4 61 ...
 $ LM411_R1_t3: num  6246 0 2012 23 66 ...
 $ LM411_R1_t4: num  10693 0 3293 372 80 ...
 $ LM411_R2_t0: num  6594 0 54 0 25 ...
 $ LM411_R2_t1: num  7304 0 689 0 43 ...
 $ LM411_R2_t2: num  7481 0 2088 10 73 ...
 $ LM411_R2_t3: num  6583 0 1804 41 61 ...
 $ LM411_R2_t4: num  8048 0 2464 348 72 ...
 $ LM411_R3_t0: num  6290 0 69 0 17 ...
 $ LM411_R3_t1: num  8712 0 734 1 54 ...
 $ LM411_R3_t2: num  6660 0 1780 7 91 ...
 $ LM411_R3_t3: num  7215 0 2030 66 99 ...
 $ LM411_R3_t4: num  8932 0 2720 387 69 ...
 $ LM511_R1_t0: num  4556 0 49 0 6 ...
 $ LM511_R1_t1: num  7442 0 823 0 84 ...
 $ LM511_R1_t2: num  7753 0 1920 5 61 ...
 $ LM511_R1_t3: num  8931 0 2166 19 90 ...
 $ LM511_R1_t4: num  7866 0 2080 6

In [5]:
all_raw_counts$gene_id<-gsub("\\..+$","",all_raw_counts$gene_id)

str(all_raw_counts)
cat("\n")

'data.frame':	78334 obs. of  31 variables:
 $ gene_id    : chr  "ENSMUSG00000000001" "ENSMUSG00000000003" "ENSMUSG00000000028" "ENSMUSG00000000031" ...
 $ LM411_R1_t0: num  5384 0 58 0 16 ...
 $ LM411_R1_t1: num  8039 0 912 0 53 ...
 $ LM411_R1_t2: num  6724 0 1953 4 61 ...
 $ LM411_R1_t3: num  6246 0 2012 23 66 ...
 $ LM411_R1_t4: num  10693 0 3293 372 80 ...
 $ LM411_R2_t0: num  6594 0 54 0 25 ...
 $ LM411_R2_t1: num  7304 0 689 0 43 ...
 $ LM411_R2_t2: num  7481 0 2088 10 73 ...
 $ LM411_R2_t3: num  6583 0 1804 41 61 ...
 $ LM411_R2_t4: num  8048 0 2464 348 72 ...
 $ LM411_R3_t0: num  6290 0 69 0 17 ...
 $ LM411_R3_t1: num  8712 0 734 1 54 ...
 $ LM411_R3_t2: num  6660 0 1780 7 91 ...
 $ LM411_R3_t3: num  7215 0 2030 66 99 ...
 $ LM411_R3_t4: num  8932 0 2720 387 69 ...
 $ LM511_R1_t0: num  4556 0 49 0 6 ...
 $ LM511_R1_t1: num  7442 0 823 0 84 ...
 $ LM511_R1_t2: num  7753 0 1920 5 61 ...
 $ LM511_R1_t3: num  8931 0 2166 19 90 ...
 $ LM511_R1_t4: num  7866 0 2080 61 66 ...
 $ LM511

In [6]:
multiVals <- function(x) paste(x,collapse=";")

In [7]:
all_raw_counts$gene_name <- mapIds(org.Mm.eg.db, keys=all_raw_counts$gene_id, keytype="ENSEMBL",column="SYMBOL", multiVals=multiVals)

str(all_raw_counts)
cat("\n")

'select()' returned 1:many mapping between keys and columns



'data.frame':	78334 obs. of  32 variables:
 $ gene_id    : chr  "ENSMUSG00000000001" "ENSMUSG00000000003" "ENSMUSG00000000028" "ENSMUSG00000000031" ...
 $ LM411_R1_t0: num  5384 0 58 0 16 ...
 $ LM411_R1_t1: num  8039 0 912 0 53 ...
 $ LM411_R1_t2: num  6724 0 1953 4 61 ...
 $ LM411_R1_t3: num  6246 0 2012 23 66 ...
 $ LM411_R1_t4: num  10693 0 3293 372 80 ...
 $ LM411_R2_t0: num  6594 0 54 0 25 ...
 $ LM411_R2_t1: num  7304 0 689 0 43 ...
 $ LM411_R2_t2: num  7481 0 2088 10 73 ...
 $ LM411_R2_t3: num  6583 0 1804 41 61 ...
 $ LM411_R2_t4: num  8048 0 2464 348 72 ...
 $ LM411_R3_t0: num  6290 0 69 0 17 ...
 $ LM411_R3_t1: num  8712 0 734 1 54 ...
 $ LM411_R3_t2: num  6660 0 1780 7 91 ...
 $ LM411_R3_t3: num  7215 0 2030 66 99 ...
 $ LM411_R3_t4: num  8932 0 2720 387 69 ...
 $ LM511_R1_t0: num  4556 0 49 0 6 ...
 $ LM511_R1_t1: num  7442 0 823 0 84 ...
 $ LM511_R1_t2: num  7753 0 1920 5 61 ...
 $ LM511_R1_t3: num  8931 0 2166 19 90 ...
 $ LM511_R1_t4: num  7866 0 2080 61 66 ...
 $ LM511

## Remove all NA gene symbol

In [8]:
indx.NA<-which(all_raw_counts$gene_name == "NA")

str(indx.NA)
cat("\n")

all_raw_counts<-all_raw_counts[-indx.NA,]

str(all_raw_counts)
cat("\n")

 int [1:45482] 161 1098 1433 2166 2583 2624 2795 2859 2915 3153 ...

'data.frame':	32852 obs. of  32 variables:
 $ gene_id    : chr  "ENSMUSG00000000001" "ENSMUSG00000000003" "ENSMUSG00000000028" "ENSMUSG00000000031" ...
 $ LM411_R1_t0: num  5384 0 58 0 16 ...
 $ LM411_R1_t1: num  8039 0 912 0 53 ...
 $ LM411_R1_t2: num  6724 0 1953 4 61 ...
 $ LM411_R1_t3: num  6246 0 2012 23 66 ...
 $ LM411_R1_t4: num  10693 0 3293 372 80 ...
 $ LM411_R2_t0: num  6594 0 54 0 25 ...
 $ LM411_R2_t1: num  7304 0 689 0 43 ...
 $ LM411_R2_t2: num  7481 0 2088 10 73 ...
 $ LM411_R2_t3: num  6583 0 1804 41 61 ...
 $ LM411_R2_t4: num  8048 0 2464 348 72 ...
 $ LM411_R3_t0: num  6290 0 69 0 17 ...
 $ LM411_R3_t1: num  8712 0 734 1 54 ...
 $ LM411_R3_t2: num  6660 0 1780 7 91 ...
 $ LM411_R3_t3: num  7215 0 2030 66 99 ...
 $ LM411_R3_t4: num  8932 0 2720 387 69 ...
 $ LM511_R1_t0: num  4556 0 49 0 6 ...
 $ LM511_R1_t1: num  7442 0 823 0 84 ...
 $ LM511_R1_t2: num  7753 0 1920 5 61 ...
 $ LM511_R1_t3: num  8931

## Force all NAs 0

In [9]:
all_raw_counts[is.na(all_raw_counts)] <- 0

## Convert into a DESeq2 count matrix

In [10]:
all_raw_counts_matrix<-as.matrix(all_raw_counts[,-c(which(colnames(all_raw_counts) == 'gene_id'),which(colnames(all_raw_counts) == 'gene_name'))])

row.names(all_raw_counts_matrix)<-all_raw_counts$gene_name


cat("all_raw_counts_matrix_0\n")
str(all_raw_counts_matrix)
cat("\n")

all_raw_counts_matrix_0
 num [1:32852, 1:30] 5384 0 58 0 16 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:32852] "Gnai3" "Pbsn" "Cdc45" "H19" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



# Keep to integers

In [11]:
all_raw_counts_matrix <- round(all_raw_counts_matrix, digits = 0)


cat("all_raw_counts_matrix_1\n")
str(all_raw_counts_matrix)
cat("\n")

all_raw_counts_matrix_1
 num [1:32852, 1:30] 5384 0 58 0 16 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:32852] "Gnai3" "Pbsn" "Cdc45" "H19" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



## Check some expression values

In [14]:
row.names(all_raw_counts_matrix)[grep("Stat1", row.names(all_raw_counts_matrix))]

[1] "Stat1"

In [15]:
all_raw_counts_matrix[which(row.names(all_raw_counts_matrix) == 'Stat1'),]

LM411_R1_t0 LM411_R1_t1 LM411_R1_t2 LM411_R1_t3 LM411_R1_t4 LM411_R2_t0 
      22347       15759       19514       18932       26057       27163 
LM411_R2_t1 LM411_R2_t2 LM411_R2_t3 LM411_R2_t4 LM411_R3_t0 LM411_R3_t1 
      14035       22203       19169       21023       21711       15820 
LM411_R3_t2 LM411_R3_t3 LM411_R3_t4 LM511_R1_t0 LM511_R1_t1 LM511_R1_t2 
      19754       20802       25445       15320       12844       23025 
LM511_R1_t3 LM511_R1_t4 LM511_R2_t0 LM511_R2_t1 LM511_R2_t2 LM511_R2_t3 
      23125       17866       23434       13393       24607       20563 
LM511_R2_t4 LM511_R3_t0 LM511_R3_t1 LM511_R3_t2 LM511_R3_t3 LM511_R3_t4 
      28770       18341       13540       21276       22896       21142

## Count matrix

In [16]:
count_matrix<-all_raw_counts_matrix

cat("count_matrix_0\n")
cat(str(count_matrix))
cat("\n")

count_matrix_0
 num [1:32852, 1:30] 5384 0 58 0 16 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:32852] "Gnai3" "Pbsn" "Cdc45" "H19" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



# Create the colData object

In [17]:
samples<-colnames(count_matrix)

cat("samples/n")
cat(str(samples))


samples/n chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...


In [18]:
treat<-gsub("_.+$","",samples)

cat("treat/n")
cat(str(treat))


treat/n chr [1:30] "LM411" "LM411" "LM411" "LM411" "LM411" "LM411" "LM411" "LM411" ...


In [19]:
batch<-gsub("^[^_]+_","",samples)
batch<-gsub("_.+$","",batch)


cat("batch/n")
cat(str(batch))


batch/n chr [1:30] "R1" "R1" "R1" "R1" "R1" "R2" "R2" "R2" "R2" "R2" "R3" "R3" ...


In [20]:
time<-gsub("^[^_]+_[^_]+_","",samples)

cat("time/n")
cat(str(time))


time/n chr [1:30] "t0" "t1" "t2" "t3" "t4" "t0" "t1" "t2" "t3" "t4" "t0" "t1" ...


In [22]:
colData<-as.data.frame(cbind(samples,treat,batch,time))

colnames(colData)<-c('sample','treat','batch','time')

cat("colData_0/n")
cat(str(colData))


colData_0/n'data.frame':	30 obs. of  4 variables:
 $ sample: chr  "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...
 $ treat : chr  "LM411" "LM411" "LM411" "LM411" ...
 $ batch : chr  "R1" "R1" "R1" "R1" ...
 $ time  : chr  "t0" "t1" "t2" "t3" ...


## factorize treat

In [88]:
colData$treat<-factor(colData$treat)
colData$treat<-relevel(colData$treat, ref='LM411') ### Nothing works with ordered factors

colData$time<-factor(colData$time)
colData$time<-relevel(colData$time, ref='t0') ### Nothing works with ordered factors

cat("colData_1/n")
cat(str(colData))

colData_1/n'data.frame':	30 obs. of  4 variables:
 $ sample: chr  "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...
 $ treat : Factor w/ 2 levels "LM411","LM511": 1 1 1 1 1 1 1 1 1 1 ...
 $ batch : chr  "R1" "R1" "R1" "R1" ...
 $ time  : Factor w/ 5 levels "t0","t1","t2",..: 1 2 3 4 5 1 2 3 4 5 ...


# DESeq2

## Create the DESeqDataSetFromMatrix object

In [89]:
dds <- DESeqDataSetFromMatrix(countData = count_matrix,
                              colData = colData,
                              design = ~ treat + time)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“51 duplicate rownames were renamed by adding numbers”


In [90]:
cat("dds_0/n")
#cat(str(dds))

dds_0/n

## Filter out genes with fewer than  total raw counts

In [91]:
keep <- rowSums(counts(dds)) >= 50

filtered_out<-keep[which(keep == "FALSE")]

cat(str(filtered_out))

 Named logi [1:18006] FALSE FALSE FALSE FALSE FALSE FALSE ...
 - attr(*, "names")= chr [1:18006] "Pbsn" "Apoh" "Zfy2" "Ngfr" ...


In [92]:
grep("Stat1",names(filtered_out))

integer(0)

In [93]:
dds <- dds[keep,]

## estimateSizeFactors----------------------------------


In [94]:
dds<-estimateSizeFactors(dds)

## LRT test from the reduced model

In [95]:
dds_lrt <- DESeq(dds, 
                 test = "LRT", 
                 reduced = ~ time)

using pre-existing size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



## Extract results

In [96]:
possible_contrasts<-colnames(dds_lrt@modelMatrix)[-1] # -1 because 1 is the Intercept term

possible_contrasts

[1] "treat_LM511_vs_LM411" "time_t1_vs_t0"        "time_t2_vs_t0"       
[4] "time_t3_vs_t0"        "time_t4_vs_t0"

In [97]:
resultsNames(dds_lrt)

[1] "Intercept"            "treat_LM511_vs_LM411" "time_t1_vs_t0"       
[4] "time_t2_vs_t0"        "time_t3_vs_t0"        "time_t4_vs_t0"

In [98]:
tmp_results<-results(dds_lrt, test= "Wald", name=possible_contrasts[1], independentFiltering=FALSE)
        
#### expand LogFC #########################################

tmp_results <- lfcShrink(dds_lrt, 
                         coef = possible_contrasts[1],
                         res=tmp_results,
                         type = "apeglm")


cat("tmp_results_0/n")
#cat(str(tmp_results))

using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895



tmp_results_0/n

In [99]:
 #### obtain data frame #########################################
        
tmp_tb <- as.data.frame(tmp_results %>%
                          data.frame() %>%
                          rownames_to_column(var = "gene") %>%
                          as_tibble() %>%
                          arrange(padj), stringsAsFactors=F)

tmp_tb$contrast<-possible_contrasts[1]

cat("tmp_tb_0")
cat(str(tmp_tb))

tmp_tb_0'data.frame':	14846 obs. of  7 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...


In [100]:
tmp_tb$minuslog10padj <- -log10(tmp_tb$padj)
      
tmp_tb$abslogfc<-abs(tmp_tb$log2FoldChange)

cat("tmp_tb_0")
cat(str(tmp_tb))

tmp_tb_0'data.frame':	14846 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...


In [101]:
DE_results<-tmp_tb

## Extract normalized counts

In [102]:
nor_counts<-as.matrix(counts(dds, normalized=TRUE))
      
row.names(nor_counts)<-row.names(nor_counts)

cat("nor_counts_0\n")
str(nor_counts)
cat("\n")


nor_counts_0
 num [1:14846, 1:30] 7030.9 75.7 0 20.9 1648 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:14846] "Gnai3" "Cdc45" "H19" "Scml2" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



# SAVE results

In [268]:
setwd("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/")

In [269]:
write.table(DE_results, file="DE_results.tsv", sep="\t", quote=F, row.names=F)

write.table(nor_counts, file="nor_counts.tsv", sep="\t", quote=F)


## Explore results

In [106]:
SIG<-DE_results[which(DE_results$minuslog10padj >= 3),]

cat("SIG_0\n")
str(SIG)
cat("\n")

SIG_0
'data.frame':	2517 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...



# Heatmap

## Load libraries

In [107]:
.libPaths()
assign(".lib.loc", "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library", envir = environment(.libPaths))
.libPaths()
# # sessionInfo()


# .libPaths()
# .libPaths(new = c("/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"))
# .libPaths()
# sessionInfo()

library("optparse")
suppressMessages(library(dplyr)) 
suppressMessages(library(ggplot2)) 
suppressMessages(library(Matrix)) 
suppressMessages(library(data.table)) 
suppressMessages(library(ggpubr)) 
suppressMessages(library(ggplot2))
suppressMessages(library(pheatmap))
suppressMessages(library("cowplot"))
suppressMessages(library("RColorBrewer"))
suppressMessages(library("plyr"))
suppressMessages(library("forcats"))
suppressMessages(library('ggeasy'))
suppressMessages(library('dplyr'))
suppressMessages(library("svglite"))
suppressMessages(library("ape"))
suppressMessages(library("ggforce"))
suppressMessages(library("tidyr"))
suppressMessages(library("tibble")) 
suppressMessages(library(future))
library("ggrepel")



[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

# Create graph path

In [122]:
if(file.exists("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/DE_graphs")){

        path_graphs<-"/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/DE_graphs"

}else{

    dir.create("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/DE_graphs")
    path_graphs<-"/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/DE_graphs"
}

## Read in data

In [279]:
setwd("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/")

In [280]:
DE_results<-read.table(file="DE_results.tsv", sep="\t", header=T)

cat("DE_results\n")
str(DE_results)
cat("\n")

DE_results
'data.frame':	14846 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...



In [125]:
normalised_counts<-as.matrix(read.table(file="nor_counts.tsv", sep="\t", header=T))

cat("normalised_counts\n")
str(normalised_counts)
cat("\n")

normalised_counts
 num [1:14846, 1:30] 7030.9 75.7 0 20.9 1648 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:14846] "Gnai3" "Cdc45" "H19" "Scml2" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



In [126]:
SIG<-DE_results[which(DE_results$minuslog10padj >= 1.3),]

cat("SIG_0\n")
str(SIG)
cat("\n")

SIG_0
'data.frame':	4850 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...



## Represent DE genes or if there are not the top 75 

In [127]:
DE_results<-DE_results[order(DE_results$minuslog10padj, decreasing = TRUE),]


cat("DE_results_1\n")
str(DE_results)
cat("\n")



DE_results_1
'data.frame':	14846 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...



In [128]:
selected<-DE_results[c(1:75),]

cat("selected_0\n")
str(selected)
cat("\n")

selected_0
'data.frame':	75 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...



In [129]:
DEBUG <- 1

In [130]:
 gene_EXP_matrix<-normalised_counts[which(row.names(normalised_counts)%in%selected$gene),]
       
       if(DEBUG == 1){
         
         cat("gene_EXP_matrix_0\n")
         cat(str(gene_EXP_matrix))
         cat("\n")
       }

gene_EXP_matrix_0
 num [1:75, 1:30] 2.61 0 0 2.61 13.06 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:75] "Ccl3" "Col5a3" "Pgf" "Zbtb32" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



## annotation_col

In [136]:
 annotation_col<- data.frame(matrix(vector(), length(colnames(gene_EXP_matrix)), 3,
                                          dimnames=list(c(),
                                                        c("treat","Batch",'time'))),stringsAsFactors=F)
       
       row.names(annotation_col)<-colnames(gene_EXP_matrix)
       

       
       if(DEBUG == 1){
         cat("annotation_col_0\n")
         cat(str(annotation_col))
         cat("\n")
         cat(str(row.names(annotation_col)))
         cat("\n")
         
       }
       
       
       annotation_col$treat<-gsub("_.+$","", row.names(annotation_col))
       annotation_col$Batch<-gsub("^[^_]+_","", row.names(annotation_col))
       annotation_col$Batch<-gsub("_.+$","", annotation_col$Batch)
       annotation_col$time<-gsub("^[^_]+_[^_]+_","", row.names(annotation_col))


    
       if(DEBUG == 1){
         cat("annotation_col_1\n")
         cat(str(annotation_col))
         cat("\n")
       }

 annotation_col$treat<-factor(annotation_col$treat,
                                       levels=c('LM411','LM511'),
                                       ordered=T)

annotation_col$Batch<-factor(annotation_col$Batch,
                                       levels=c('R1','R2','R3'),
                                       ordered=T)

      if(DEBUG == 1){
         cat("annotation_col_2\n")
         cat(str(annotation_col))
         cat("\n")
       }
       
      

annotation_col_0
'data.frame':	30 obs. of  3 variables:
 $ treat: logi  NA NA NA NA NA NA ...
 $ Batch: logi  NA NA NA NA NA NA ...
 $ time : logi  NA NA NA NA NA NA ...

 chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...

annotation_col_1
'data.frame':	30 obs. of  3 variables:
 $ treat: chr  "LM411" "LM411" "LM411" "LM411" ...
 $ Batch: chr  "R1" "R1" "R1" "R1" ...
 $ time : chr  "t0" "t1" "t2" "t3" ...

annotation_col_2
'data.frame':	30 obs. of  3 variables:
 $ treat: Ord.factor w/ 2 levels "LM411"<"LM511": 1 1 1 1 1 1 1 1 1 1 ...
 $ Batch: Ord.factor w/ 3 levels "R1"<"R2"<"R3": 1 1 1 1 1 2 2 2 2 2 ...
 $ time : chr  "t0" "t1" "t2" "t3" ...



## ann colors

In [141]:
vector_colors_Batch<- c(brewer.pal(9, "Greens")[c(5,7,9)])

        names(vector_colors_Batch)<-levels(annotation_col$Batch)

vector_colors_Batch
        
        vector_colors_treat<-c(brewer.pal(9, "Greens")[c(5)],                                 
                                    brewer.pal(9, "Reds")[c(5)])
        
        names(vector_colors_treat)<-levels(annotation_col$treat)

vector_colors_treat

R1        R2        R3 
"#74C476" "#238B45" "#00441B"

LM411     LM511 
"#74C476" "#FB6A4A"

In [142]:
ann_colors <- list( Batch = vector_colors_Batch,
                           treat = vector_colors_treat)
      
       
       if(DEBUG == 1){
         cat("ann_colors_0\n")
         cat(str(ann_colors))
         cat("\n")
       }

ann_colors_0
List of 2
 $ Batch: Named chr [1:3] "#74C476" "#238B45" "#00441B"
  ..- attr(*, "names")= chr [1:3] "R1" "R2" "R3"
 $ treat: Named chr [1:2] "#74C476" "#FB6A4A"
  ..- attr(*, "names")= chr [1:2] "LM411" "LM511"



## Actual heatmap

In [143]:
heatmap<-pheatmap(gene_EXP_matrix, display_numbers = FALSE,
                             show_colnames=FALSE,
                             angle_col = "0",
                             clustering_method="ward.D2",
                             fontsize_row = 6, 
                             fontsize_col = 8,
                             breaks=seq(-2,2,length.out=101),
                             color=colorRampPalette(c("blue","white","red"))(100),
                             scale="row",
                             cluster_cols=TRUE,
                             border_color='black',
                             treeheight_row=70, treeheight_col=70, cutree_cols=3,
                             annotation_col = annotation_col,
                             annotation_colors = ann_colors)



ERROR: Error in value[[3L]](cond): no active device and default getOption("device") is invalid


plot without title

In [144]:
setwd(path_graphs)
         
svgname<-paste("Heatmap_genes",".svg",sep='')

ggsave(svgname,plot=heatmap, device ='svg')

Saving 7 x 7 in image


# Volcano plot

In [62]:
cat("DE_results_1\n")
str(DE_results)
cat("\n")

DE_results_1
'data.frame':	16938 obs. of  9 variables:
 $ gene          : chr  "ATP6V0D2" "JAG1" "CDH1" "S100A9" ...
 $ baseMean      : num  4004 1376 221 2387 2432 ...
 $ log2FoldChange: num  0.913 0.913 2.084 -0.855 -0.731 ...
 $ lfcSE         : num  0.133 0.163 0.388 0.171 0.149 ...
 $ pvalue        : num  3.42e-13 1.00e-09 3.30e-09 2.30e-08 3.78e-08 ...
 $ padj          : num  5.79e-09 8.50e-06 1.86e-05 9.74e-05 1.28e-04 ...
 $ contrast      : chr  "condition_rs35592432_vs_wt" "condition_rs35592432_vs_wt" "condition_rs35592432_vs_wt" "condition_rs35592432_vs_wt" ...
 $ minuslog10padj: num  8.24 5.07 4.73 4.01 3.89 ...
 $ abslogfc      : num  0.913 0.913 2.084 0.855 0.731 ...



## Actual volcano plot

In [158]:
 volcano_plot<-ggplot(data=DE_results,
                                aes(x=log2FoldChange,
                                    y=minuslog10padj)) +
             geom_vline(xintercept=c(-0.1,0.1), color="gray", linetype='dashed',linewidth=1)+
             geom_hline(yintercept=c(1.3), color="gray", linetype='dashed',linewidth=1)+
             geom_point(color="gray",fill="gray", stroke=0.2, shape=21, size=2)+
             geom_point(data=DE_results[which(DE_results$minuslog10padj >= 6 & DE_results$log2FoldChange <= -2),],
                        color="black",fill="blue", stroke=0.2, shape=21, size=4)+
             geom_point(data=DE_results[which(DE_results$minuslog10padj >= 6 & DE_results$log2FoldChange >= 2),],
                        color="black",fill="red", stroke=0.2, shape=21, size=4)+
             scale_y_continuous(name="-log10 pval")+
             ggtitle(paste(dim(SIG)[1], "DE genes", sep=" "))+
             theme_classic()+
             theme(plot.title=element_text(color="black", family="sans", size=14),
                   axis.title.y=element_text(color="black", family="sans", size=12),
                   axis.title.x=element_text(color="black", family="sans", size=12),
                   axis.text.y=element_text(color="black", family="sans", size=8),
                   axis.text.x=element_text(color="black", family="sans", size=8))+
             theme(legend.title = element_text(family="sans"),
                   legend.text = element_text(family="sans"),
                   legend.key.size = unit(0.35, 'cm'), #change legend key size
                   legend.key.height = unit(0.35, 'cm'), #change legend key height
                   legend.key.width = unit(0.35, 'cm'), #change legend key width
                   legend.position="hidden")+
             ggeasy::easy_center_title()
           
           volcano_plot <-volcano_plot+
             geom_text_repel(data=DE_results[which(DE_results$minuslog10padj >= 12 & DE_results$log2FoldChange <= -2),],
                             aes(x=log2FoldChange,
                                 y=minuslog10padj,
                                 label=gene),
                             family="sans",
                             fontface='italic',
                             color='blue',
                             segment.size  = 0.25,
                             segment.color = "blue",
                             force=25,
                             size=4,
                             box.padding = 1,
                             max.overlaps = Inf)
           
           volcano_plot <-volcano_plot+
             geom_text_repel(data=DE_results[which(DE_results$minuslog10padj >= 12
                                                   & DE_results$log2FoldChange >= 2),],
                             aes(x=log2FoldChange,
                                 y=minuslog10padj,
                                 label=gene),
                             family="sans",
                             fontface='italic',
                             color='red',
                             segment.size  = 0.25,
                             segment.color = "red",
                             force=25,
                             size=4,
                             box.padding = 1,
                             max.overlaps = Inf)



In [159]:
setwd(path_graphs)
         
svgname<-paste("Volcano_genes",".svg",sep='')

ggsave(svgname,plot=volcano_plot, device ='svg')

Saving 7 x 7 in image
Warning message:
“Removed 30 rows containing missing values (`geom_point()`).”


# Load ORA analysis libraries

In [160]:
.libPaths()
.libPaths(new = c("/home/manuel.tardaguila/conda_envs/GSEA/lib/R/library"))
.libPaths()

suppressMessages(library(clusterProfiler))
suppressMessages(library(enrichplot))
suppressMessages(library(tidyverse))
#suppressMessages(library(ggupset))
suppressMessages(library(RColorBrewer))
suppressMessages(library(pheatmap))
suppressMessages(library(DOSE))
suppressMessages(library("splitstackshape"))
suppressMessages(library('data.table'))
suppressMessages(library("optparse"))
suppressMessages(library('cowplot'))
suppressMessages(library("org.Mm.eg.db"))


[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] "/home/manuel.tardaguila/conda_envs/GSEA/lib/R/library"                   
[2] "/group/soranzo/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

In [161]:
multiVals <- function(x) paste(x,collapse=";")

# Pick-up and ORA analysis

In [162]:
setwd("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/")

In [163]:
DE_results<-read.table(file="DE_results.tsv", sep="\t", header=T)

cat("DE_results\n")
cat(str(DE_results))
cat("\n")

nor_counts<-read.table(file="nor_counts.tsv", sep="\t", header=T)

cat("nor_counts\n")
cat(str(nor_counts))
cat("\n")

DE_results
'data.frame':	14846 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...

nor_counts
'data.frame':	14846 obs. of  30 variables:
 $ LM411_R1_t0: num  7030.9 75.7 0 20.9 1648 ...
 $ LM411_R1_t1: num  7194.1 816.2 0 47.4 827.8 ...
 $ LM411_R1_t2: num  6598.42 1916.53 3.93 59.86 973.47 ...
 $ LM411_R1_t3: num  6758.3 2177 24.9 71.4 937 ...
 $ LM411_R1_t4: num  7568.6 2330.8 263.3 56.6 996.6 ...
 $ LM411_R2_t0: num  6485

In [164]:
out_path <- paste0("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/",'/','ORA','_','background_adapted','/') # output path, where you want your results exported to

  if(file.exists(out_path)){

    unlink(out_path, recursive =T)

    dir.create(out_path)
  }else{

    dir.create(out_path)
  }


  cat("out_path_\n")
  cat(sprintf(as.character(out_path)))
  cat("\n")

out_path_
/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted/


In [169]:
 #### READ and transform path_to_GMT ----
  
path_to_GMT = '/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs/'

cat("path_to_GMT_0\n")
cat(sprintf(as.character(path_to_GMT)))
cat("\n")

path_to_GMT_0
/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs/


# Search terms

In [195]:
 #### READ and transform search_terms ----
  
search_terms = unlist(strsplit('LAMININ,IMMUNITY,IMMUNE_RESPONSE,T_CELL,SENESCENCE', split=","))

cat("search_terms_\n")
cat(sprintf(as.character(search_terms)))
cat("\n")


#### READ and transform pval_threshold ----

pval_threshold = 0.05

cat("pval_threshold_\n")
cat(sprintf(as.character(pval_threshold)))
cat("\n")

search_terms_
LAMININ IMMUNITY IMMUNE_RESPONSE T_CELL SENESCENCE
pval_threshold_
0.05


In [196]:
### Filter out NA padj -------------------------------
  
DE_results_NO_NA<-DE_results[!is.na(DE_results$padj),]

cat("DE_results_NO_NA_0\n")
cat(str(DE_results_NO_NA))
cat("\n")

DE_results_NO_NA$ENTREZID <- mapIds(org.Mm.eg.db, keys=DE_results_NO_NA$gene, keytype="SYMBOL",
                                      column="ENTREZID", multiVals=multiVals)

cat("DE_results_NO_NA_1\n")
str(DE_results_NO_NA)
cat("\n")

DE_results_NO_NA_with_ENTREZ<-DE_results_NO_NA[-which(DE_results_NO_NA$ENTREZID == "NA"),]

cat("DE_results_NO_NA_with_ENTREZ_0\n")
str(DE_results_NO_NA_with_ENTREZ)
cat("\n")

DE_results_NO_NA_0
'data.frame':	14816 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...



'select()' returned 1:1 mapping between keys and columns



DE_results_NO_NA_1
'data.frame':	14816 obs. of  10 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...
 $ ENTREZID      : chr  "16323" "17022" "23859" "21983" ...

DE_results_NO_NA_with_ENTREZ_0
'data.frame':	14725 obs. of  10 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.14

In [197]:
## Get the genes that are present in your dataframe -----------------------------------------
  
dataset_genes_df<-as.data.frame(unique(DE_results_NO_NA_with_ENTREZ$ENTREZID), stringsAsFactors=F)
colnames(dataset_genes_df)<-'ENTREZID'

cat("dataset_genes_df_0\n")
cat(str(dataset_genes_df))
cat("\n")

gmt_files <- list.files(path = path_to_GMT, pattern = 'Mm.entrez.gmt$', full.names = TRUE)

cat("gmt_files_0\n")
cat(str(gmt_files))
cat("\n")

gmt_files

dataset_genes_df_0
'data.frame':	14725 obs. of  1 variable:
 $ ENTREZID: chr  "16323" "17022" "23859" "21983" ...

gmt_files_0
 chr [1:19] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m1.all.v2"| __truncated__ ...



[1] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m1.all.v2023.2.Mm.entrez.gmt"            
 [2] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m2.all.v2023.2.Mm.entrez.gmt"            
 [3] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m2.cgp.v2023.2.Mm.entrez.gmt"            
 [4] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m2.cp.biocarta.v2023.2.Mm.entrez.gmt"    
 [5] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m2.cp.reactome.v2023.2.Mm.entrez.gmt"    
 [6] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m2.cp.v2023.2.Mm.entrez.gmt"             
 [7] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m2.cp.wikipathways.v2023.2.Mm.entrez.gmt"
 [8] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m3.all.v2023.2.Mm.entrez.gmt"            
 [9] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m3.gtrd.v2023.2.Mm.entrez.gmt"           
[10] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m3.mirdb.v2023.2.Mm.entrez.gmt"          
[11] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m5.all.v2023.2.Mm.entrez.gmt"            
[12] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m5.go.bp.v2023.2.Mm.entrez.gmt"          
[13] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m5.go.cc.v2023.2.Mm.entrez.gmt"          
[14] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m5.go.mf.v2023.2.Mm.entrez.gmt"          
[15] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m5.go.v2023.2.Mm.entrez.gmt"             
[16] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m5.mpt.v2023.2.Mm.entrez.gmt"            
[17] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//m8.all.v2023.2.Mm.entrez.gmt"            
[18] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//mh.all.v2023.2.Mm.entrez.gmt"            
[19] "/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//msigdb.v2023.2.Mm.entrez.gmt"

#  ####### LOOP 1 save all the background files a---------------------------------------------------------

In [198]:
  
  DEBUG<-0
  
  universe_set<-'m5.go.v2023.2.Mm.entrez.gmt'
  
  for (iteration_gmt_files in 1:length(gmt_files)){
    
    file <- gmt_files[iteration_gmt_files]
    filename<-gsub(paste0(path_to_GMT,'/'),"",file)
    
    cat("------------------------------------------->\t")
    cat(sprintf(as.character(filename)))
    cat("\n")
    
    
    complete_collection <- read.gmt(file)
    
    if(DEBUG ==1){
      cat("complete_collection_0\n")
      cat(str(complete_collection))
      cat("\n")
    }
    
    
    
    complete_collection_gene_sel <- complete_collection[which(complete_collection$gene %in% dataset_genes_df$ENTREZID),]
    
    if(DEBUG ==1){
      cat("complete_collection_gene_sel_0\n")
      cat(str(complete_collection_gene_sel))
      cat("\n")
    }
    
    
    ### set universe -------------------
    
    
    FLAG_universe_set<-sum(filename == universe_set)
    
    if(FLAG_universe_set == 1){
      
      universe<-complete_collection_gene_sel
      
      filename_universe<-gsub("\\.gmt$","_universe.rds",filename)
    
      ## SAVE universe
      
      setwd(out_path)
      
      saveRDS(universe, file=filename_universe) 
      
    }else{
      
      
    }#FLAG_universe_set == 1
    
    
    FLAG_custom<-length(grep("Custom_Soranzo_",filename))
    
    cat("FLAG_custom\t")
    cat(sprintf(as.character(FLAG_custom)))
    cat("\n")
    
    if(FLAG_custom == 0){
      
      
      matches <- grep(paste(search_terms,collapse="|"),complete_collection_gene_sel$term)
      
      toMatch<-tolower(search_terms)
      
      
      matches_lc <- grep(paste(toMatch,collapse="|"),complete_collection_gene_sel$term)
      
      total_matches<-unique(c(matches,matches_lc))               
      
      
      if(length(total_matches) > 1){
        
        complete_collection_gene_sel_term_sel<-droplevels(complete_collection_gene_sel[total_matches,])
        
        
        if(DEBUG ==1){
          
          cat("complete_collection_gene_sel_term_sel_0\n")
          str(complete_collection_gene_sel_term_sel)
          cat("\n")
        }
        
        
        filename<-gsub("\\.gmt$","_selected.rds",filename)
        
        
        
        #filename <- paste('test','_',i,'.rds', sep='') #paste(gsub('c.\\.', '', gsub('.v7.5.*$', '', file)), '.RDS', sep = '')
        
        cat(sprintf(as.character(filename)))
        cat("\n")
        cat(sprintf(as.character(out_path)))
        cat("\n")
        
        ## SAVE 1
        
        setwd(out_path)
        
        saveRDS(complete_collection_gene_sel_term_sel, file=filename) 
        
        
        equivalence_df<-complete_collection_gene_sel_term_sel
        
        ### 2 equivalence
        
        equivalence_df$SYMBOL <- mapIds(org.Mm.eg.db, keys=equivalence_df$gene, keytype="ENTREZID",
                                        column="SYMBOL", multiVals=multiVals)
        
        
        equivalence_df<-equivalence_df[,-which(colnames(equivalence_df) == 'gene')]
        
        colnames(equivalence_df)[which(colnames(equivalence_df) == 'SYMBOL')]<-'gene'
        
        
        equivalence_df.dt<-data.table(equivalence_df, key="term")
        
        
        equivalence_df_collapsed<-as.data.frame(equivalence_df.dt[,.(genes=paste(gene, collapse=",")), by=key(equivalence_df.dt)], stringsAsFactors=F)
        
        if(DEBUG ==1){
          
          cat("equivalence_df_collapsed_0\n")
          str(equivalence_df_collapsed)
          cat("\n")
        }
        
        
        filename_2<-gsub("_selected\\.rds$","_selected_equivalence.tsv",filename)
        
        
        #filename_2 <- paste('test','_',i,'.rds', sep='') #paste(gsub('c.\\.', '', gsub('.v7.5.*$', '', file)), '.RDS', sep = '')
        
        cat(sprintf(as.character(filename_2)))
        cat("\n")
        cat(sprintf(as.character(out_path)))
        cat("\n")
        
        
        setwd(out_path)
        
        write.table(equivalence_df_collapsed, file=filename_2, sep = "\t",quote=F,row.names = F) 
        
      }#length(total_matches) > 1
      
    }else{
      
      complete_collection_gene_sel_term_sel<-complete_collection_gene_sel
      
      
      if(DEBUG ==1){
        
        cat("complete_collection_gene_sel_term_sel_0\n")
        str(complete_collection_gene_sel_term_sel)
        cat("\n")
      }
      
      
      filename<-gsub("\\.gmt$","_selected.rds",filename)
      
      
      
      #filename <- paste('test','_',i,'.rds', sep='') #paste(gsub('c.\\.', '', gsub('.v7.5.*$', '', file)), '.RDS', sep = '')
      
      cat(sprintf(as.character(filename)))
      cat("\n")
      cat(sprintf(as.character(out_path)))
      cat("\n")
      
      ## SAVE 1
      
      setwd(out_path)
      
      saveRDS(complete_collection_gene_sel_term_sel, file=filename) 
      
      
      equivalence_df<-complete_collection_gene_sel_term_sel
      
      ### 2 equivalence
      
      equivalence_df$SYMBOL <- mapIds(org.Mm.eg.db, keys=equivalence_df$gene, keytype="ENTREZID",
                                      column="SYMBOL", multiVals=multiVals)
      
      
      equivalence_df<-equivalence_df[,-which(colnames(equivalence_df) == 'gene')]
      
      colnames(equivalence_df)[which(colnames(equivalence_df) == 'SYMBOL')]<-'gene'
      
      
      equivalence_df.dt<-data.table(equivalence_df, key="term")
      
      
      equivalence_df_collapsed<-as.data.frame(equivalence_df.dt[,.(genes=paste(gene, collapse=",")), by=key(equivalence_df.dt)], stringsAsFactors=F)
      
      if(DEBUG ==1){
        
        cat("equivalence_df_collapsed_0\n")
        str(equivalence_df_collapsed)
        cat("\n")
      }
      
      
      filename_2<-gsub("_selected\\.rds$","_selected_equivalence.tsv",filename)
      
      
      #filename_2 <- paste('test','_',i,'.rds', sep='') #paste(gsub('c.\\.', '', gsub('.v7.5.*$', '', file)), '.RDS', sep = '')
      
      cat(sprintf(as.character(filename_2)))
      cat("\n")
      cat(sprintf(as.character(out_path)))
      cat("\n")
      
      
      setwd(out_path)
      
      write.table(equivalence_df_collapsed, file=filename_2, sep = "\t",quote=F,row.names = F) 
      
    }#FLAG_custom == 0
    
   
    
    
    
 
  }#iteration_gmt_files in 1:length(gmt_files)

------------------------------------------->	m1.all.v2023.2.Mm.entrez.gmt
FLAG_custom	0
------------------------------------------->	m2.all.v2023.2.Mm.entrez.gmt
FLAG_custom	0
m2.all.v2023.2.Mm.entrez_selected.rds
/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted/


'select()' returned 1:1 mapping between keys and columns



m2.all.v2023.2.Mm.entrez_selected_equivalence.tsv
/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted/
------------------------------------------->	m2.cgp.v2023.2.Mm.entrez.gmt
FLAG_custom	0
m2.cgp.v2023.2.Mm.entrez_selected.rds
/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted/


'select()' returned 1:1 mapping between keys and columns



m2.cgp.v2023.2.Mm.entrez_selected_equivalence.tsv
/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted/
------------------------------------------->	m2.cp.biocarta.v2023.2.Mm.entrez.gmt
FLAG_custom	0
------------------------------------------->	m2.cp.reactome.v2023.2.Mm.entrez.gmt
FLAG_custom	0
m2.cp.reactome.v2023.2.Mm.entrez_selected.rds
/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted/


...

m8.all.v2023.2.Mm.entrez_selected_equivalence.tsv
/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted/
------------------------------------------->	mh.all.v2023.2.Mm.entrez.gmt
FLAG_custom	0
------------------------------------------->	msigdb.v2023.2.Mm.entrez.gmt


Warning message in readLines(gmtfile):
“line 16091 appears to contain an embedded nul”
Warning message in readLines(gmtfile):
“incomplete final line found on '/home/manuel.tardaguila/GMT_files/msigdb_v2023.2.Mm_files_to_download_locally/msigdb_v2023.2.Mm_GMTs//msigdb.v2023.2.Mm.entrez.gmt'”


FLAG_custom	0
msigdb.v2023.2.Mm.entrez_selected.rds
/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted/


'select()' returned 1:1 mapping between keys and columns



msigdb.v2023.2.Mm.entrez_selected_equivalence.tsv
/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted/


#  #### Restrict to DE genes ----

In [208]:
DE_results_NO_NA_SIG<-DE_results_NO_NA_with_ENTREZ[which(DE_results_NO_NA_with_ENTREZ$minuslog10padj >= 1.3), ]

cat("DE_results_NO_NA_SIG_0\n")
str(DE_results_NO_NA_SIG)
cat("\n")

DE_results_NO_NA_SIG_0
'data.frame':	4831 obs. of  10 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...
 $ ENTREZID      : chr  "16323" "17022" "23859" "21983" ...



# LOOP ORA

In [209]:
options(enrichment_force_universe=TRUE)


DEBUG<-0

ORA_df<-data.frame()

universe_set<-'m5.go.v2023.2.Mm.entrez_selected.rds'

In [210]:
bg_files <- list.files(path = out_path, pattern = '_selected.rds$', full.names = TRUE)

cat("bg_files_0\n")
str(bg_files)
cat("\n")

bg_files_0
 chr [1:14] "/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted//m2.all.v2023.2.Mm"| __truncated__ ...



## universe

In [211]:
 for(iteration_bg_files in 1:length(bg_files)){
          
          
          
          selected_collection<-bg_files[iteration_bg_files]
          
          cat("--------------------------------------------------------------------->\t")
          cat(sprintf(as.character(iteration_bg_files)))
          cat("\t")
          cat(sprintf(as.character(selected_collection)))
          cat("\n")
          
          selected_collection_df<-readRDS(file=selected_collection)
          
          if(DEBUG ==1){
            
            cat("selected_collection_df_0\n")
            # str(selected_collection_df)
            cat("\n")
          }
          
          setwd(out_path)
          
          universe_df<-readRDS(file=universe_set)
          universe<-unique(universe_df$gene)
          
            
          cat("universe_set_of_unique_genes\n")
          str(unique(universe_df$gene))
          cat("\n")
        
          
          
          FLAG_custom<-length(grep("Custom_Soranzo_", selected_collection))
          
          
          
          cat("FLAG_custom_0\n")
          str(FLAG_custom)
          cat("\n")
          
          if(FLAG_custom == 0){
            
            minGSSize_spec<-10
            maxGSSize_spec<-500
            DEBUG<-0
            
            
          }else{
            
            setwd(out_path)
            
            
           
            minGSSize_spec<-1
            maxGSSize_spec<-1000
            
            DEBUG<-1
            
            if(DEBUG ==1){
              
              cat("selected_collection_df_0\n")
              str(selected_collection_df)
              cat("\n")
            }
            # universe<-unique(selected_collection_df$gene)
            
          }

     res<-enricher(gene= DE_results_NO_NA_SIG$ENTREZID, 
                                           TERM2GENE = selected_collection_df, 
                                           universe=universe,
                                           maxGSSize=maxGSSize_spec,
                                           minGSSize=minGSSize_spec,
                                           pvalueCutoff = 0.05,
                                           pAdjustMethod = "BH",
                                           qvalueCutoff = 0.25)

     

      if(DEBUG ==1){
              
              cat("res_0\n")
              str(res)
              cat("\n")
            }

     FLAG_NULL<-sum(is.null(res))



        if(FLAG_NULL == 0){

                res_df <- res@result                   
                
                if(DEBUG ==1){
                  
                  cat("res_df_0\n")
                  str(res_df)
                  cat("\n")
                }

              res_df <- res_df %>% mutate(minuslog10padj = -log10(p.adjust),
                                                diffexpressed = gsub('\\..+$', '', rownames(res_df)))                    
                    
                    if(DEBUG ==1){
                      
                      cat("res_df_1\n")
                      str(res_df)
                      cat("\n")
                    }
        
             res_df_long<-unique(as.data.frame(cSplit(res_df,sep = '/', direction = "long",
                                                               splitCols = "geneID"),stringsAsFactors=F))
                      
                      res_df_long$geneID<-as.character(res_df_long$geneID)
                      
                      if(DEBUG ==1){
                        str(res_df_long)
                        cat("\n")
                      }
                      
                      res_df_long$geneID <- mapIds(org.Mm.eg.db, keys=res_df_long$geneID, keytype="ENTREZID",
                                                   column="SYMBOL", multiVals=multiVals)
                      
                      if(DEBUG ==1){
                        str(res_df_long)
                        cat("\n")
                      }
                      
                      res_df_long.dt<-data.table(res_df_long, key=colnames(res_df_long)[-which(colnames(res_df_long) == 'geneID')])
                      
                      
                      res_df_long_collapsed<-as.data.frame(res_df_long.dt[,.(geneID=paste(geneID, collapse='/')), by=key(res_df_long.dt)], stringsAsFactors=F)
                      
                      if(DEBUG ==1){
                        str(res_df_long_collapsed)
                        cat("\n")
                      }
        
             ORA_df<-rbind(ORA_df,res_df_long_collapsed)
        
        }#FLAG_NULL == 0    

     
}#iteration_bg_files in 1:length(bg_files)

--------------------------------------------------------------------->	1	/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted//m2.all.v2023.2.Mm.entrez_selected.rds
universe_set_of_unique_genes
 chr [1:1970] "11350" "11596" "12367" "60533" "12524" "54698" "13848" ...

FLAG_custom_0
 int 0



Warning message in type.convert.default(unlist(x, use.names = FALSE)):
“'as.is' should be specified by the caller; using TRUE”
'select()' returned 1:1 mapping between keys and columns



--------------------------------------------------------------------->	2	/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted//m2.cgp.v2023.2.Mm.entrez_selected.rds
universe_set_of_unique_genes
 chr [1:1970] "11350" "11596" "12367" "60533" "12524" "54698" "13848" ...

FLAG_custom_0
 int 0



Warning message in type.convert.default(unlist(x, use.names = FALSE)):
“'as.is' should be specified by the caller; using TRUE”
'select()' returned 1:1 mapping between keys and columns



--------------------------------------------------------------------->	3	/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted//m2.cp.biocarta.v2023.2.Mm.entrez_selected.rds
universe_set_of_unique_genes
 chr [1:1970] "11350" "11596" "12367" "60533" "12524" "54698" "13848" ...

FLAG_custom_0
 int 0



...

Warning message in type.convert.default(unlist(x, use.names = FALSE)):
“'as.is' should be specified by the caller; using TRUE”
'select()' returned 1:1 mapping between keys and columns



--------------------------------------------------------------------->	13	/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted//m8.all.v2023.2.Mm.entrez_selected.rds
universe_set_of_unique_genes
 chr [1:1970] "11350" "11596" "12367" "60533" "12524" "54698" "13848" ...

FLAG_custom_0
 int 0



Warning message in type.convert.default(unlist(x, use.names = FALSE)):
“'as.is' should be specified by the caller; using TRUE”
'select()' returned 1:1 mapping between keys and columns



--------------------------------------------------------------------->	14	/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025//ORA_background_adapted//msigdb.v2023.2.Mm.entrez_selected.rds
universe_set_of_unique_genes
 chr [1:1970] "11350" "11596" "12367" "60533" "12524" "54698" "13848" ...

FLAG_custom_0
 int 0



Warning message in type.convert.default(unlist(x, use.names = FALSE)):
“'as.is' should be specified by the caller; using TRUE”
'select()' returned 1:1 mapping between keys and columns



# Results

In [212]:
str(ORA_df)

'data.frame':	857 obs. of  14 variables:
 $ ID            : chr  "GALINDO_IMMUNE_RESPONSE_TO_ENTEROTOXIN" "REACTOME_CELLULAR_SENESCENCE" "REACTOME_OXIDATIVE_STRESS_INDUCED_SENESCENCE" "WP_WHITE_FAT_CELL_DIFFERENTIATION" ...
 $ Description   : chr  "GALINDO_IMMUNE_RESPONSE_TO_ENTEROTOXIN" "REACTOME_CELLULAR_SENESCENCE" "REACTOME_OXIDATIVE_STRESS_INDUCED_SENESCENCE" "WP_WHITE_FAT_CELL_DIFFERENTIATION" ...
 $ GeneRatio     : chr  "30/111" "10/111" "6/111" "8/111" ...
 $ BgRatio       : chr  "38/1970" "23/1970" "13/1970" "19/1970" ...
 $ RichFactor    : num  0.789 0.435 0.462 0.421 0.789 ...
 $ FoldEnrichment: num  14.01 7.72 8.19 7.47 24.69 ...
 $ zScore        : num  19.79 7.92 6.36 6.93 26.79 ...
 $ pvalue        : num  1.96e-32 1.33e-07 3.47e-05 3.56e-06 1.79e-41 ...
 $ p.adjust      : num  7.82e-32 2.67e-07 3.47e-05 4.74e-06 1.79e-41 ...
 $ qvalue        : num  NA NA NA NA NA NA NA NA NA NA ...
 $ Count         : int  30 10 6 8 30 8 10 6 10 6 ...
 $ minuslog10padj: num  31.11 6.57 4.4

In [213]:
ORA_df_SIG<-ORA_df[which(ORA_df$p.adjust <= pval_threshold & ORA_df$Count >=3),]

cat("ORA_df_SIG_0\n")
str(ORA_df_SIG)
cat("\n")
cat(sprintf(as.character(names(summary(as.factor(ORA_df_SIG$identity))))))
cat("\n")
cat(sprintf(as.character(summary(as.factor(ORA_df_SIG$identity)))))
cat("\n")
cat(sprintf(as.character(names(summary(as.factor(ORA_df_SIG$contrast))))))
cat("\n")
cat(sprintf(as.character(summary(as.factor(ORA_df_SIG$contrast)))))
cat("\n")

ORA_df_SIG_0
'data.frame':	41 obs. of  14 variables:
 $ ID            : chr  "GALINDO_IMMUNE_RESPONSE_TO_ENTEROTOXIN" "REACTOME_CELLULAR_SENESCENCE" "REACTOME_OXIDATIVE_STRESS_INDUCED_SENESCENCE" "WP_WHITE_FAT_CELL_DIFFERENTIATION" ...
 $ Description   : chr  "GALINDO_IMMUNE_RESPONSE_TO_ENTEROTOXIN" "REACTOME_CELLULAR_SENESCENCE" "REACTOME_OXIDATIVE_STRESS_INDUCED_SENESCENCE" "WP_WHITE_FAT_CELL_DIFFERENTIATION" ...
 $ GeneRatio     : chr  "30/111" "10/111" "6/111" "8/111" ...
 $ BgRatio       : chr  "38/1970" "23/1970" "13/1970" "19/1970" ...
 $ RichFactor    : num  0.789 0.435 0.462 0.421 0.789 ...
 $ FoldEnrichment: num  14.01 7.72 8.19 7.47 24.69 ...
 $ zScore        : num  19.79 7.92 6.36 6.93 26.79 ...
 $ pvalue        : num  1.96e-32 1.33e-07 3.47e-05 3.56e-06 1.79e-41 ...
 $ p.adjust      : num  7.82e-32 2.67e-07 3.47e-05 4.74e-06 1.79e-41 ...
 $ qvalue        : num  NA NA NA NA NA NA NA NA NA NA ...
 $ Count         : int  30 10 6 8 30 8 10 6 10 6 ...
 $ minuslog10padj: num  31

In [214]:
ORA_df_SIG$ID

[1] "GALINDO_IMMUNE_RESPONSE_TO_ENTEROTOXIN"                                        
 [2] "REACTOME_CELLULAR_SENESCENCE"                                                  
 [3] "REACTOME_OXIDATIVE_STRESS_INDUCED_SENESCENCE"                                  
 [4] "WP_WHITE_FAT_CELL_DIFFERENTIATION"                                             
 [5] "GALINDO_IMMUNE_RESPONSE_TO_ENTEROTOXIN"                                        
 [6] "BIOCARTA_ASBCELL_PATHWAY"                                                      
 [7] "REACTOME_CELLULAR_SENESCENCE"                                                  
 [8] "REACTOME_OXIDATIVE_STRESS_INDUCED_SENESCENCE"                                  
 [9] "REACTOME_CELLULAR_SENESCENCE"                                                  
[10] "REACTOME_OXIDATIVE_STRESS_INDUCED_SENESCENCE"                                  
[11] "WP_WHITE_FAT_CELL_DIFFERENTIATION"                                             
[12] "WP_WHITE_FAT_CELL_DIFFERENTIATION"                                             
[13] "GOBP_T_CELL_ACTIVATION"                                                        
[14] "GOBP_ADAPTIVE_IMMUNE_RESPONSE"                                                 
[15] "GOBP_ALPHA_BETA_T_CELL_ACTIVATION"                                             
[16] "GOBP_ALPHA_BETA_T_CELL_DIFFERENTIATION"                                        
[17] "GOBP_CD4_POSITIVE_ALPHA_BETA_T_CELL_ACTIVATION"                                
[18] "GOBP_IMMUNE_RESPONSE_REGULATING_CELL_SURFACE_RECEPTOR_SIGNALING_PATHWAY"       
[19] "GOBP_IMMUNE_RESPONSE_REGULATING_SIGNALING_PATHWAY"                             
[20] "GOBP_INNATE_IMMUNE_RESPONSE_ACTIVATING_CELL_SURFACE_RECEPTOR_SIGNALING_PATHWAY"
[21] "GOBP_NEGATIVE_REGULATION_OF_ALPHA_BETA_T_CELL_ACTIVATION"                      
[22] "GOBP_REGULATION_OF_T_CELL_ACTIVATION"                                          
[23] "GOBP_T_CELL_ACTIVATION"                                                        
[24] "GOBP_T_CELL_DIFFERENTIATION"                                                   
[25] "GOCC_T_CELL_RECEPTOR_COMPLEX"                                                  
[26] "GOMF_LAMININ_BINDING"                                                          
[27] "GOMF_T_CELL_RECEPTOR_BINDING"                                                  
[28] "GOBP_ALPHA_BETA_T_CELL_ACTIVATION"                                             
[29] "GOBP_IMMUNE_RESPONSE_REGULATING_CELL_SURFACE_RECEPTOR_SIGNALING_PATHWAY"       
[30] "GOBP_T_CELL_ACTIVATION"                                                        
[31] "GOBP_T_CELL_DIFFERENTIATION"                                                   
[32] "MP_INCREASED_T_CELL_DERIVED_LYMPHOMA_INCIDENCE"                                
[33] "TABULA_MURIS_SENIS_KIDNEY_T_CELL_AGEING"                                       
[34] "TABULA_MURIS_SENIS_LIMB_MUSCLE_T_CELL_AGEING"                                  
[35] "TABULA_MURIS_SENIS_LUNG_CD4_POSITIVE_ALPHA_BETA_T_CELL_AGEING"                 
[36] "TABULA_MURIS_SENIS_LUNG_CD8_POSITIVE_ALPHA_BETA_T_CELL_AGEING"                 
[37] "TABULA_MURIS_SENIS_MAMMARY_GLAND_T_CELL_AGEING"                                
[38] "TABULA_MURIS_SENIS_MARROW_MATURE_ALPHA_BETA_T_CELL_AGEING"                     
[39] "TABULA_MURIS_SENIS_SPLEEN_CD4_POSITIVE_ALPHA_BETA_T_CELL_AGEING"               
[40] "TABULA_MURIS_SENIS_SPLEEN_CD8_POSITIVE_ALPHA_BETA_T_CELL_AGEING"               
[41] "TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING"

# SAVE

In [270]:
setwd("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/")

In [271]:
write.table(ORA_df_SIG, file=paste("ORA_results_DE_significant",".tsv",sep=''), sep="\t", quote=F, row.names = F)


# Pick-up and graphs

In [272]:
setwd("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/")

In [273]:
DE_results<-read.table(file="DE_results.tsv", sep="\t", header=T)

cat("DE_results\n")
cat(str(DE_results))
cat("\n")

nor_counts<-read.table(file="nor_counts.tsv", sep="\t", header=T)

cat("nor_counts\n")
cat(str(nor_counts))
cat("\n")


ORA_df_SIG<-read.table(file=paste("ORA_results_DE_significant",".tsv",sep=''), sep="\t", header=T)

cat("ORA_df_SIG_0\n")
cat(str(ORA_df_SIG))
cat("\n")


DE_results
'data.frame':	14846 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...

nor_counts
'data.frame':	14846 obs. of  30 variables:
 $ LM411_R1_t0: num  7030.9 75.7 0 20.9 1648 ...
 $ LM411_R1_t1: num  7194.1 816.2 0 47.4 827.8 ...
 $ LM411_R1_t2: num  6598.42 1916.53 3.93 59.86 973.47 ...
 $ LM411_R1_t3: num  6758.3 2177 24.9 71.4 937 ...
 $ LM411_R1_t4: num  7568.6 2330.8 263.3 56.6 996.6 ...
 $ LM411_R2_t0: num  6485

# Select TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING target genes

In [224]:
indx.int<-grep("TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING", ORA_df_SIG$ID)

str(indx.int)
cat("\n")

 int 41



In [225]:
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_DE_targets<-unlist(strsplit(ORA_df_SIG$geneID[indx.int], split='/'))

str(TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_DE_targets)
cat("\n")

 chr [1:57] "Bcl2a1d" "Id2" "Cxcr3" "Pdcd1" "Tnfrsf4" "Cd44" "Lime1" ...



# Create graph path

In [226]:
if(file.exists("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_graphs/")){


}else{

    dir.create("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_graphs/")
}

        path_graphs<-"/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_graphs/"


## Volcano plot highlight TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_DE_targets

In [232]:
 volcano_plot<-ggplot(data=DE_results,
                                aes(x=log2FoldChange,
                                    y=minuslog10padj)) +
             geom_vline(xintercept=c(-0.25,0.25), color="gray", linetype='dashed',linewidth=1)+
             geom_hline(yintercept=c(1.3), color="gray", linetype='dashed',linewidth=1)+
             geom_point(color="gray",fill="gray", stroke=0.2, shape=21, size=2)+
             geom_point(data=DE_results[which(DE_results$minuslog10padj >= 3 & DE_results$log2FoldChange <= -0.5),],
                        color="black",fill="blue", stroke=0.2, shape=21, size=4)+
             geom_point(data=DE_results[which(DE_results$minuslog10padj >= 3 & DE_results$log2FoldChange >= 0.5),],
                        color="black",fill="red", stroke=0.2, shape=21, size=4)+ 
             scale_y_continuous(name="-log10 pval")+
             theme_classic()+
             theme(plot.title=element_text(color="black", family="sans", size=14),
                   axis.title.y=element_text(color="black", family="sans", size=12),
                   axis.title.x=element_text(color="black", family="sans", size=12),
                   axis.text.y=element_text(color="black", family="sans", size=8),
                   axis.text.x=element_text(color="black", family="sans", size=8))+
             theme(legend.title = element_text(family="sans"),
                   legend.text = element_text(family="sans"),
                   legend.key.size = unit(0.35, 'cm'), #change legend key size
                   legend.key.height = unit(0.35, 'cm'), #change legend key height
                   legend.key.width = unit(0.35, 'cm'), #change legend key width
                   legend.position="hidden")+
             ggeasy::easy_center_title()
           
           volcano_plot <-volcano_plot+
             geom_text_repel(data=DE_results[which(DE_results$gene%in%TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_DE_targets & DE_results$log2FoldChange <= -0.5),],
                             aes(x=log2FoldChange,
                                 y=minuslog10padj,
                                 label=gene),
                             family="sans",
                             fontface='italic',
                             color='blue',
                             segment.size  = 0.25,
                             segment.color = "blue",
                             force=25,
                             size=4,
                             box.padding = 1,
                             max.overlaps = Inf)
           
           volcano_plot <-volcano_plot+
             geom_text_repel(data=DE_results[which(DE_results$gene%in%TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_DE_targets & DE_results$log2FoldChange >= 0.5),],
                             aes(x=log2FoldChange,
                                 y=minuslog10padj,
                                 label=gene),
                             family="sans",
                             fontface='italic',
                             color='red',
                             segment.size  = 0.25,
                             segment.color = "red",
                             force=25,
                             size=4,
                             box.padding = 1,
                             max.overlaps = Inf)

#              ggtitle(paste(paste(paste(contrast_sel, sep=''), paste(identity_sel, sep=''),sep='  '),paste(dim(SIG_genes)[1], "DE genes", sep=" "), sep="\n"))+


volcano_plot

Warning message:
“Removed 30 rows containing missing values (`geom_point()`).”


ERROR: Error in value[[3L]](cond): no active device and default getOption("device") is invalid


plot without title

In [233]:
setwd(path_graphs)
         
svgname<-paste("Volcano_TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets",".svg",sep='')

ggsave(svgname,plot=volcano_plot, device ='svg')

Saving 7 x 7 in image
Warning message:
“Removed 30 rows containing missing values (`geom_point()`).”


# Heatmap

## Load libraries

In [234]:
.libPaths()
assign(".lib.loc", "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library", envir = environment(.libPaths))
.libPaths()
# # sessionInfo()


# .libPaths()
# .libPaths(new = c("/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"))
# .libPaths()
# sessionInfo()

library("optparse")
suppressMessages(library(dplyr)) 
suppressMessages(library(ggplot2)) 
suppressMessages(library(Matrix)) 
suppressMessages(library(data.table)) 
suppressMessages(library(ggpubr)) 
suppressMessages(library(ggplot2))
suppressMessages(library(pheatmap))
suppressMessages(library("cowplot"))
suppressMessages(library("RColorBrewer"))
suppressMessages(library("plyr"))
suppressMessages(library("forcats"))
suppressMessages(library('ggeasy'))
suppressMessages(library('dplyr'))
suppressMessages(library("svglite"))
suppressMessages(library("ape"))
suppressMessages(library("ggforce"))
suppressMessages(library("tidyr"))
suppressMessages(library("tibble")) 
suppressMessages(library(future))
library("ggrepel")



[1] "/home/manuel.tardaguila/conda_envs/GSEA/lib/R/library"                   
[2] "/group/soranzo/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

# Create graph path

In [235]:
if(file.exists("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/DE_graphs")){

        path_graphs<-"/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/DE_graphs"

}else{

    dir.create("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/DE_graphs")
    path_graphs<-"/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/DE_graphs"
}

## Read in data

In [274]:
setwd("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/")

In [275]:
DE_results<-read.table(file="DE_results.tsv", sep="\t", header=T)

cat("DE_results\n")
str(DE_results)
cat("\n")

DE_results
'data.frame':	14846 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...



In [276]:
normalised_counts<-as.matrix(read.table(file="nor_counts.tsv", sep="\t", header=T))

cat("normalised_counts\n")
str(normalised_counts)
cat("\n")

normalised_counts
 num [1:14846, 1:30] 7030.9 75.7 0 20.9 1648 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:14846] "Gnai3" "Cdc45" "H19" "Scml2" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



In [277]:
ORA_df_SIG<-read.table(file=paste("ORA_results_DE_significant",".tsv",sep=''), sep="\t", header=T)

cat("ORA_df_SIG_0\n")
cat(str(ORA_df_SIG))
cat("\n")


ORA_df_SIG_0
'data.frame':	41 obs. of  14 variables:
 $ ID            : chr  "GALINDO_IMMUNE_RESPONSE_TO_ENTEROTOXIN" "REACTOME_CELLULAR_SENESCENCE" "REACTOME_OXIDATIVE_STRESS_INDUCED_SENESCENCE" "WP_WHITE_FAT_CELL_DIFFERENTIATION" ...
 $ Description   : chr  "GALINDO_IMMUNE_RESPONSE_TO_ENTEROTOXIN" "REACTOME_CELLULAR_SENESCENCE" "REACTOME_OXIDATIVE_STRESS_INDUCED_SENESCENCE" "WP_WHITE_FAT_CELL_DIFFERENTIATION" ...
 $ GeneRatio     : chr  "30/111" "10/111" "6/111" "8/111" ...
 $ BgRatio       : chr  "38/1970" "23/1970" "13/1970" "19/1970" ...
 $ RichFactor    : num  0.789 0.435 0.462 0.421 0.789 ...
 $ FoldEnrichment: num  14.01 7.72 8.19 7.47 24.69 ...
 $ zScore        : num  19.79 7.92 6.36 6.93 26.79 ...
 $ pvalue        : num  1.96e-32 1.33e-07 3.47e-05 3.56e-06 1.79e-41 ...
 $ p.adjust      : num  7.82e-32 2.67e-07 3.47e-05 4.74e-06 1.79e-41 ...
 $ qvalue        : num  NA NA NA NA NA NA NA NA NA NA ...
 $ Count         : int  30 10 6 8 30 8 10 6 10 6 ...
 $ minuslog10padj: num  31

In [278]:
SIG<-DE_results[which(DE_results$minuslog10padj >= 12),]

cat("SIG_0\n")
str(SIG)
cat("\n")

SIG_0
'data.frame':	349 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...



# Select TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING target genes

In [251]:
indx.int<-grep("TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING", ORA_df_SIG$ID)

str(indx.int)
cat("\n")

 int 41



In [252]:
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_DE_targets<-unlist(strsplit(ORA_df_SIG$geneID[indx.int], split='/'))

str(TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_DE_targets)
cat("\n")

 chr [1:57] "Bcl2a1d" "Id2" "Cxcr3" "Pdcd1" "Tnfrsf4" "Cd44" "Lime1" ...



In [253]:
ORA_df_SIG_TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING<-ORA_df_SIG[indx.int,]

ORA_df_SIG_TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING

,ID,Description,GeneRatio,BgRatio,RichFactor,FoldEnrichment,zScore,pvalue,p.adjust,qvalue,Count,minuslog10padj,diffexpressed,geneID
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<chr>,<chr>
41,TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING,TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING,57/817,86/1970,0.6627907,1.598161,4.773602,1.836563e-06,1.561078e-05,6.766284e-06,57,4.806575,TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING,Bcl2a1d/Id2/Cxcr3/Pdcd1/Tnfrsf4/Cd44/Lime1/S100a10/Ctss/Capg/Sdf4/Kcnn4/Gimap5/B2m/Apobec3/Icos/H2-D1/Lag3/H2-K1/Sh2d1a/Cd4/Cd3g/Tspan32/Pou2f2/Hif1a/Gimap3/Nfkbia/Actg1/H2-Q4/Il2rb/Srgn/Rac2/Ptpn22/Lgals1/Tox/Laptm5/Ndfip1/Lpxn/Cd83/Trim8/Fyn/Crlf2/Cd84/Tnfrsf1b/Tnfsf8/Lcp2/Tmbim6/Arf6/Vim/Serpina3g/Calr/Gpi1/Gapdh/Prelid1/Rap1a/Casp1/Zfp36l1


## Represent all DE genes and classify TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING targets

In [254]:
selected<-SIG[order(SIG$minuslog10padj, decreasing = TRUE),]

cat("selected_0\n")
str(selected)
cat("\n")

selected_0
'data.frame':	349 obs. of  9 variables:
 $ gene          : chr  "Inhba" "Lum" "Dlg2" "Tpbg" ...
 $ baseMean      : num  3031 1238 177 325 11827 ...
 $ log2FoldChange: num  2.45 2.4 3.11 2.87 2.59 ...
 $ lfcSE         : num  0.0821 0.1077 0.1401 0.1383 0.1273 ...
 $ pvalue        : num  4.95e-197 7.39e-112 1.59e-110 6.44e-99 3.37e-94 ...
 $ padj          : num  7.34e-193 5.48e-108 7.87e-107 2.39e-95 9.97e-91 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  192.1 107.3 106.1 94.6 90 ...
 $ abslogfc      : num  2.45 2.4 3.11 2.87 2.59 ...



In [255]:
DEBUG <- 1

### gene_EXP_matrix

In [256]:
 gene_EXP_matrix<-normalised_counts[which(row.names(normalised_counts)%in%selected$gene),]


       
       if(DEBUG == 1){
         
         cat("gene_EXP_matrix_0\n")
         cat(str(gene_EXP_matrix))
         cat("\n")
       }

gene_EXP_matrix_0
 num [1:349, 1:30] 24140.74 10.45 344.76 94.02 2.61 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:349] "Itgb2" "Pdgfb" "Itga5" "Il12rb1" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



## Annotation_row

In [257]:
df<-as.data.frame(cbind(row.names(gene_EXP_matrix),rep(NA,length(row.names(gene_EXP_matrix)))), stringsAsFactors=F)

colnames(df)<-c('gene_name','CLASS')


cat("df_0\n")
cat(str(df))
cat("\n")

df$CLASS[which(df$gene_name%in%TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_DE_targets)]<-'TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets'

summary(as.factor(df$CLASS))


df_0
'data.frame':	349 obs. of  2 variables:
 $ gene_name: chr  "Itgb2" "Pdgfb" "Itga5" "Il12rb1" ...
 $ CLASS    : chr  NA NA NA NA ...



TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                             10 
                                           NA's 
                                            339

In [258]:
annotation_row = data.frame(GeneClass = df$CLASS)
      
      rownames(annotation_row) = df$gene_name
      
      if(DEBUG == 1){
        cat("annotation_row_0\n")
        cat(str(annotation_row))
        cat("\n")
        cat(sprintf(as.character(names(summary(as.factor(annotation_row$GeneClass))))))
        cat("\n")
        cat(sprintf(as.character(summary(as.factor(annotation_row$GeneClass)))))
        cat("\n")
           cat(sprintf(as.character(row.names(annotation_row))))
        cat("\n")
      }

annotation_row_0
'data.frame':	349 obs. of  1 variable:
 $ GeneClass: chr  NA NA NA NA ...

TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets NA's
10 339
Itgb2 Pdgfb Itga5 Il12rb1 Ccl3 S100a3 Ifrd1 Il16 Ikzf4 Dusp3 Man1a Stat5a Col5a3 Angptl2 Pgf Cd44 Ndrg1 Slc30a4 Tnfsf14 Zbtb32 Fhl2 Elk3 Kdsr Podnl1 St3gal1 Clip3 Glis2 Kif1a Ttbk1 Eps8 Rxra Cd5l Dstn Ncf1 Cd274 Pdcd1lg2 Dyrk3 Igfbp4 Rab11fip4 Ctsa Cyth4 Ikzf3 Chd3 Mmp7 Dusp14 Myh11 Cxcl16 Ccl4 Ahr Cd70 Perp Sgk1 Socs2 Pno1 Plek Egfr Slc1a4 Adora2a Mdm1 Adarb1 Gfpt2 Itk Trib2 Gna13 Id2 Grhl1 Pecam1 Rhbdf2 Cacna1g Snx11 Nr1d1 Myh10 Asb2 Nedd9 Susd3 Pde4d Serinc5 Rnf180 Fam107a Atp8a2;LOC108168164 Tnfsf11 Fyb Myo10 Slc25a17 Ly6e Cd200 Ccdc80 St3gal6 Donson Ifnar2 Enah Galnt14 Mcfd2 Ltb Tnf Ms4a6b Slc15a3 Cemip2 Ehd1 Pip5k1b Add3 Ablim1 Ints4 Hexa Farp1 Shisa5 Hspa4l Slco3a1 Nrp2 Raph1 Casp8 Ccl20 Obsl1 Pdcd1 Lrrfip1 Rgs2 Rgs16 Lamc2 Cdc42bpa Atf3 Tnfsf4 Niban2 St6galnac6 Ak1 Nr4a2 Lamc3 Zeb2 Il1rn Nckap1 Ehd4 Rassf2 Il1a Samhd1 Il2 Dclk1 

## annotation_col

In [259]:
 annotation_col<- data.frame(matrix(vector(), length(colnames(gene_EXP_matrix)), 3,
                                          dimnames=list(c(),
                                                        c("treat","Batch",'time'))),stringsAsFactors=F)
       
       row.names(annotation_col)<-colnames(gene_EXP_matrix)
       

       
       if(DEBUG == 1){
         cat("annotation_col_0\n")
         cat(str(annotation_col))
         cat("\n")
         cat(str(row.names(annotation_col)))
         cat("\n")
         
       }
       
       
       annotation_col$treat<-gsub("_.+$","", row.names(annotation_col))
       annotation_col$Batch<-gsub("^[^_]+_","", row.names(annotation_col))
       annotation_col$Batch<-gsub("_.+$","", annotation_col$Batch)
       annotation_col$time<-gsub("^[^_]+_[^_]+_","", row.names(annotation_col))


    
       if(DEBUG == 1){
         cat("annotation_col_1\n")
         cat(str(annotation_col))
         cat("\n")
       }

 annotation_col$treat<-factor(annotation_col$treat,
                                       levels=c('LM411','LM511'),
                                       ordered=T)

annotation_col$Batch<-factor(annotation_col$Batch,
                                       levels=c('R1','R2','R3'),
                                       ordered=T)

      if(DEBUG == 1){
         cat("annotation_col_2\n")
         cat(str(annotation_col))
         cat("\n")
       }
       

annotation_col_0
'data.frame':	30 obs. of  3 variables:
 $ treat: logi  NA NA NA NA NA NA ...
 $ Batch: logi  NA NA NA NA NA NA ...
 $ time : logi  NA NA NA NA NA NA ...

 chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...

annotation_col_1
'data.frame':	30 obs. of  3 variables:
 $ treat: chr  "LM411" "LM411" "LM411" "LM411" ...
 $ Batch: chr  "R1" "R1" "R1" "R1" ...
 $ time : chr  "t0" "t1" "t2" "t3" ...

annotation_col_2
'data.frame':	30 obs. of  3 variables:
 $ treat: Ord.factor w/ 2 levels "LM411"<"LM511": 1 1 1 1 1 1 1 1 1 1 ...
 $ Batch: Ord.factor w/ 3 levels "R1"<"R2"<"R3": 1 1 1 1 1 2 2 2 2 2 ...
 $ time : chr  "t0" "t1" "t2" "t3" ...



## ann colors

In [260]:
vector_colors_Batch<- c(brewer.pal(9, "Greens")[c(5,7,9)])

        names(vector_colors_Batch)<-levels(annotation_col$Batch)

vector_colors_Batch
        
        vector_colors_treat<-c(brewer.pal(9, "Greens")[c(5)],                                 
                                    brewer.pal(9, "Reds")[c(5)])
        
        names(vector_colors_treat)<-levels(annotation_col$treat)

vector_colors_treat

R1        R2        R3 
"#74C476" "#238B45" "#00441B"

LM411     LM511 
"#74C476" "#FB6A4A"

In [261]:
# 1. Initialize the vector with R's actual missing value (NA)
#    This allows it to be used directly in color scales that accept NA.
geneClass_vector <- rep(NA, length(annotation_row$GeneClass))

# 2. Set the names of the vector elements
names(geneClass_vector) <- annotation_row$GeneClass

# 3. Assign the color to the specific named element
#    Using "Set2" (max 8 colors) instead of "Dark2" (max 8 colors, but safer) 
#    or just using a safe hardcoded color.
#    Since you only need one color, we can safely use any palette.
geneClass_vector[which(names(geneClass_vector) == 'TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets')] <- RColorBrewer::brewer.pal(8, "Set2")[4]

# 4. View the result
#geneClass_vector

gene_class_vector_clean <- geneClass_vector[!is.na(geneClass_vector)]

gene_class_vector_clean

TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3" 
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3" 
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3" 
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3" 
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3" 
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3" 
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3" 
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3" 
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3" 
TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets 
                                      "#E78AC3"

In [262]:
ann_colors <- list( Batch = vector_colors_Batch,
                           treat = vector_colors_treat,
                  GeneClass =gene_class_vector_clean)
      
       
       if(DEBUG == 1){
         cat("ann_colors_0\n")
         cat(str(ann_colors))
         cat("\n")
       }

ann_colors_0
List of 3
 $ Batch    : Named chr [1:3] "#74C476" "#238B45" "#00441B"
  ..- attr(*, "names")= chr [1:3] "R1" "R2" "R3"
 $ treat    : Named chr [1:2] "#74C476" "#FB6A4A"
  ..- attr(*, "names")= chr [1:2] "LM411" "LM511"
 $ GeneClass: Named chr [1:10] "#E78AC3" "#E78AC3" "#E78AC3" "#E78AC3" ...
  ..- attr(*, "names")= chr [1:10] "TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets" "TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets" "TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets" "TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets" ...



## Actual heatmap

In [264]:
heatmap<-pheatmap(gene_EXP_matrix, display_numbers = FALSE,
                             show_colnames=FALSE,                  
                             angle_col = "0",
                             clustering_method="ward.D2",
                             fontsize_row = 4, 
                             fontsize_col = 8,
                             breaks=seq(-2,2,length.out=101),
                             color=colorRampPalette(c("blue","white","red"))(100),
                             scale="row",
                             cluster_cols=TRUE,
                             border_color='black',
                             treeheight_row=70, treeheight_col=70, cutree_cols=3,
                             annotation_col = annotation_col,
                             annotation_row = annotation_row,
                             annotation_colors = ann_colors)



ERROR: Error in value[[3L]](cond): no active device and default getOption("device") is invalid


plot without title

In [266]:
setwd(path_graphs)
         
svgname<-paste("Heatmap_with_TABULA_MURIS_SENIS_SPLEEN_T_CELL_AGEING_targets",".svg",sep='')

ggsave(svgname,plot=heatmap, device ='svg', height=12, width =13)